In [24]:
numbers = list(range(1000))
print(numbers[:10])


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [25]:
numbers = list(range(1000))
print(numbers[:10])

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [26]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/employees.csv")

df.head()

,id,name,department,salary,city
0,1,Arun,Sales,30000,Chennai
1,2,Priya,HR,45000,Chennai
2,3,Rahul,IT,60000,Bangalore
3,4,Divya,IT,75000,Chennai
4,5,Karthik,Finance,90000,Mumbai


In [27]:
%timeit [x*x for x in numbers]

41.4 μs ± 474 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [28]:
print("Dataset Shape:", df.shape)
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

Dataset Shape: (8, 5)
Total Rows: 8
Total Columns: 5


In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          8 non-null      int64
 1   name        8 non-null      str  
 2   department  8 non-null      str  
 3   salary      8 non-null      int64
 4   city        8 non-null      str  
dtypes: int64(2), str(3)
memory usage: 452.0 bytes


In [30]:
print("STATISTICAL SUMMARY")
display(df.describe(include="all"))

print("MISSING VALUES")
display(df.isna().sum())

print("DUPLICATE RECORDS")
print(df.duplicated().sum())

STATISTICAL SUMMARY


,id,name,department,salary,city
count,8.00000,8,8,8.000000,8
unique,NaN,8,4,NaN,3
top,NaN,Arun,IT,NaN,Chennai
freq,NaN,1,3,NaN,4
mean,4.50000,NaN,NaN,58500.000000,NaN
std,2.44949,NaN,NaN,18913.336187,NaN
min,1.00000,NaN,NaN,30000.000000,NaN
25%,2.75000,NaN,NaN,47250.000000,NaN
50%,4.50000,NaN,NaN,56000.000000,NaN
75%,6.25000,NaN,NaN,69750.000000,NaN


MISSING VALUES


id            0
name          0
department    0
salary        0
city          0
dtype: int64

DUPLICATE RECORDS
0


print("Dataset Shape:", df.shape)

## Business Questions

1. Which department has the highest number of employees?
2. Which city has the highest number of employees?
3. Which department has the highest average salary?
4. Who is the highest-paid employee?
5. What is the average salary of all employees?

# Task 3: Exploratory Data Analysis (EDA)

## 1. Loading the Cleaned Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../reports/clean_data.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset Shape: (8, 5)


,id,name,department,salary,city
0,1,Arun,Sales,30000,Chennai
1,2,Priya,HR,45000,Chennai
2,3,Rahul,IT,60000,Bangalore
3,4,Divya,IT,75000,Chennai
4,5,Karthik,Finance,90000,Mumbai


## 2. Salary Summary Statistics

Calculate the mean, median, minimum, maximum and standard deviation of employee salaries.

In [2]:
salary_stats = df["salary"].agg([
    "mean",
    "median",
    "min",
    "max",
    "std"
])

print("Employee Salary Summary Statistics:")
display(salary_stats)

Employee Salary Summary Statistics:


mean      58500.000000
median    56000.000000
min       30000.000000
max       90000.000000
std       18913.336187
Name: salary, dtype: float64

## 3. Department-wise Salary Analysis

Analyze employee count, average salary, minimum salary and maximum salary for each department using groupby().agg().

In [3]:
department_analysis = df.groupby("department").agg(
    employee_count=("id", "count"),
    average_salary=("salary", "mean"),
    minimum_salary=("salary", "min"),
    maximum_salary=("salary", "max"),
    salary_std=("salary", "std")
).round(2)

print("Department-wise Analysis:")
display(department_analysis)

Department-wise Analysis:


,employee_count,average_salary,minimum_salary,maximum_salary,salary_std
department,,,,,
Finance,1,90000.00,90000,90000,NaN
HR,2,46500.00,45000,48000,2121.32
IT,3,67666.67,60000,75000,7505.55
Sales,2,41000.00,30000,52000,15556.35


In [4]:
highest_department = department_analysis["average_salary"].idxmax()

highest_salary = department_analysis.loc[
    highest_department, "average_salary"
]

print("Department with highest average salary:", highest_department)
print("Highest average salary:", highest_salary)

Department with highest average salary: Finance
Highest average salary: 90000.0


## 4. City-wise Employee Analysis

Compare employee count, average salary and total salary across different cities using groupby().agg().

In [5]:
city_analysis = df.groupby("city").agg(
    employee_count=("id", "count"),
    average_salary=("salary", "mean"),
    total_salary=("salary", "sum")
).round(2)

print("City-wise Employee Analysis:")
display(city_analysis)

City-wise Employee Analysis:


,employee_count,average_salary,total_salary
city,,,
Bangalore,2,56000.0,112000
Chennai,4,54500.0,218000
Mumbai,2,69000.0,138000


In [6]:
highest_city = city_analysis["employee_count"].idxmax()

employee_count = city_analysis.loc[
    highest_city, "employee_count"
]

print("City with highest number of employees:", highest_city)
print("Number of employees:", employee_count)

City with highest number of employees: Chennai
Number of employees: 4


## 5. Correlation Analysis

Calculate the correlation between numerical columns using df.corr() and identify the top 3 relationships.

In [7]:
numeric_df = df.select_dtypes(include="number")

correlation_matrix = numeric_df.corr()

print("Correlation Matrix:")
display(correlation_matrix)

Correlation Matrix:


,id,salary
id,1.000000,0.357698
salary,0.357698,1.000000


In [9]:
correlation_pairs = correlation_matrix.unstack().reset_index()

correlation_pairs.columns = [
    "Column 1",
    "Column 2",
    "Correlation"
]

correlation_pairs = correlation_pairs[
    correlation_pairs["Column 1"] < correlation_pairs["Column 2"]
]

correlation_pairs = correlation_pairs.dropna()

correlation_pairs["Absolute Correlation"] = (
    correlation_pairs["Correlation"].abs()
)

top_relationships = correlation_pairs.sort_values(
    by="Absolute Correlation",
    ascending=False
)

display(top_relationships.head(3))

,Column 1,Column 2,Correlation,Absolute Correlation
1,id,salary,0.357698,0.357698


## 6. Business Questions and Answers

In [10]:
dept_avg = df.groupby("department")["salary"].mean()

print("Answer:", dept_avg.idxmax())
print("Average Salary:", dept_avg.max())

Answer: Finance
Average Salary: 90000.0


In [11]:
city_count = df["city"].value_counts()

print("Answer:", city_count.idxmax())
print("Employee Count:", city_count.max())

Answer: Chennai
Employee Count: 4


In [12]:
print("Average Salary:", df["salary"].mean())

Average Salary: 58500.0


In [13]:
top_3 = df.nlargest(3, "salary")[
    ["name", "department", "salary"]
]

display(top_3)

,name,department,salary
4,Karthik,Finance,90000
3,Divya,IT,75000
6,Suresh,IT,68000


In [14]:
dept_count = df["department"].value_counts()

print("Answer:", dept_count.idxmax())
print("Employee Count:", dept_count.max())

Answer: IT
Employee Count: 3


## 7. Department Budget Analysis

Create a department budget table and merge it with employee data using pd.merge().

In [15]:
department_budget = pd.DataFrame({
    "department": ["HR", "IT", "Finance", "Sales"],
    "budget": [200000, 400000, 300000, 350000]
})

print("Department Budget Table:")
display(department_budget)

Department Budget Table:


,department,budget
0,HR,200000
1,IT,400000
2,Finance,300000
3,Sales,350000


In [16]:
merged_df = pd.merge(
    df,
    department_budget,
    on="department",
    how="left"
)

print("Merged Employee and Budget Data:")
display(merged_df)

Merged Employee and Budget Data:


,id,name,department,salary,city,budget
0,1,Arun,Sales,30000,Chennai,350000
1,2,Priya,HR,45000,Chennai,200000
2,3,Rahul,IT,60000,Bangalore,400000
3,4,Divya,IT,75000,Chennai,400000
4,5,Karthik,Finance,90000,Mumbai,300000
5,6,Meena,Sales,52000,Bangalore,350000
6,7,Suresh,IT,68000,Chennai,400000
7,8,Anitha,HR,48000,Mumbai,200000


In [17]:
budget_analysis = merged_df.groupby("department").agg(
    employee_count=("id", "count"),
    total_salary=("salary", "sum"),
    department_budget=("budget", "first")
)

budget_analysis["remaining_budget"] = (
    budget_analysis["department_budget"] -
    budget_analysis["total_salary"]
)

print("Department Budget Analysis:")
display(budget_analysis)

Department Budget Analysis:


,employee_count,total_salary,department_budget,remaining_budget
department,,,,
Finance,1,90000,300000,210000
HR,2,93000,200000,107000
IT,3,203000,400000,197000
Sales,2,82000,350000,268000


## 8. Exporting EDA Reports